### Insurance Pricing Model

This project simulates a simplistic pricing model in insurance by calculating Premium Rates for different Ages and Policy Terms.

In [1]:
#Importing Libraries
import pandas as pd
import numpy as np
import scipy.optimize as sp
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import time

Calculating the Mortality Rate and Lapse Rate assumptions for the model

In [2]:
#Computing Mortality Rate Table
def mortality_rates():
    ages = np.arange(18, 81)
    qx = 0.0005 + (ages - 18) * 0.0001
    return pd.DataFrame({"Age": ages, "Qx": qx})

In [3]:
#Computing Lapse Rate Tale
def lapse_rates():
    durations = np.arange(1, 41)
    lapse = np.maximum(0.02 - durations * 0.0003, 0.005)
    return pd.DataFrame({"Year": durations, "Lapse_Rate": lapse})

Computing insurance cashflows and decrements 

In [4]:
#Calculating cashflows 
def cashflows(entry_age,policy_term,premium,sum_assured=100000,interest_rate=0.04):
    mort = mortality_table()
    lapse = lapse_rates()

    years = np.arange(1, policy_term + 1)
    age = entry_age + years - 1

    df = pd.DataFrame({
        "Year": years,
        "Age": age
    })

    df = df.merge(mort, on="Age", how="left")
    df = df.merge(lapse, on="Year", how="left")

    df["inforce_start"] = np.exp(-0.05 * (df["Year"] - 1))
    df["deaths"] = df["inforce_start"] * df["Qx"]
    df["lapses"] = df["inforce_start"] * df["Lapse_Rate"]

    df["claims"] = df["deaths"] * sum_assured
    df["premiums"] = df["inforce_start"] * premium

    df["net_cashflow"] = df["premiums"] - df["claims"]
    df["discount_factor"] = (1 + interest_rate) ** (-df["Year"])
    df["pv_cashflow"] = df["net_cashflow"] * df["discount_factor"]

    return df

Defining a function that returns that final output we need to optimize; the difference between the Computed Profit Margin and Target Profit Margin

In [5]:
#Calculating Profit Margin using above cashflows
def profit_margin(premium, entry_age, policy_term, target_margin):

    acq_rate = 0.35       
    renewal_rate = 0.08   
    fixed_expense = 20    
    acq_years = 3   

    years = np.arange(1, policy_term + 1)
    survival = np.exp(-0.002 * years)
    premium_cf = premium * survival
    claims_cf = 0.4 * survival * (1 + 0.02 * years)
    discount = 1 / (1.05 ** years)

    acq_expenses = np.where(years <= acq_years,acq_rate * premium / acq_years,0)
    renewal_expenses = renewal_rate * premium
    fixed_expenses = fixed_expense
    expenses_cf = (acq_expenses + renewal_expenses + fixed_expenses) * survival

    pv_premiums = np.sum(premium_cf * discount)
    pv_claims = np.sum(claims_cf * discount)
    pv_expenses = np.sum(expenses_cf * discount)

    profit = pv_premiums - pv_claims - pv_expenses
    profit_margin_actual = profit / pv_premiums

    return profit_margin_actual - target_margin

Solving the 'profit_margin' function to get the required premium

In [6]:
#Finding the roots of 'profit_margin' function
def solve_premium(entry_age, policy_term, target_margin):
    premium = sp.root(profit_margin,100,args=(entry_age, policy_term, target_margin))
    return premium.x[0]

Using parallel processing to simultaneosuly calculate multiple premium rates, using multiple cores of the CPU

In [8]:
#Creating an output table for different combinations of Ages and Policy Terms
def table_production(ages, terms, target_margin=0.05):
    results = []
    combos = [(a, t) for a in ages for t in terms]

    premiums = Parallel(n_jobs=-1)(delayed(solve_premium)(a, t, target_margin) for a, t in combos)

    for (a, t), p in zip(combos, premiums):
        results.append({"Age": a, "Policy Term": t, "Premium": p})

    return pd.DataFrame(results)

ages = range(25, 51)
terms = range(10, 31)

premium_table = table_production(ages, terms)

In [9]:
#Exporting the final table to Excel
premium_table.to_excel(r"/Users/shitikshuvyas/Desktop/Masters/Insurance_Pricing_Model_Output.xlsx")